In [ ]:
# Mount drive
from google.colab import drive
try:
    drive.mount('/content/drive')
    print("  ✓ Google Drive mounted")
except:
    print("  ✓ Google Drive already mounted")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  ✓ Google Drive mounted


In [ ]:
"""
🔒 REGRESSION-BASED TRAINING PIPELINE - FIXED VERSION
SpectroFood Apple Maturity Classification
Hybrid CNN + Transformer - Predicting Continuous Dry Matter %
✅ Proper checkpoint resuming from any epoch
"""

import os
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, Dict, List
import json

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, callbacks
from tensorflow.keras.utils import Sequence
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from google.cloud import storage

print("=" * 80)
print("SpectroFood Apple Maturity - Hybrid CNN+Transformer REGRESSION Training")
print("=" * 80)
print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

SpectroFood Apple Maturity - Hybrid CNN+Transformer REGRESSION Training
TensorFlow Version: 2.19.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================
class Config:
    """Training configuration"""
    # GCS Settings
    GCS_BUCKET = "processed_data-iyed"
    GCS_PREFIX = "processed_regression/"

    # Data files
    SPATIAL_TRAIN = "X_spatial_train.h5"
    SPATIAL_TEST = "X_spatial_test.h5"
    SPECTRAL_TRAIN = "X_spectral_train.npy"
    SPECTRAL_TEST = "X_spectral_test.npy"
    LABELS_TRAIN = "Y_train.npy"
    LABELS_TEST = "Y_test.npy"

    # Model parameters
    SPATIAL_SHAPE = (11, 11, 141)
    SPECTRAL_SHAPE = (141,)
    OUTPUT_DIM = 1

    # Training parameters
    BATCH_SIZE = 64
    EPOCHS = 100
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-5

    # CNN parameters
    CNN_FILTERS = [32, 64, 128, 256]
    CNN_DROPOUT = 0.3

    # Transformer parameters
    TRANSFORMER_HEADS = 8
    TRANSFORMER_LAYERS = 3
    TRANSFORMER_DIM = 128
    TRANSFORMER_FF_DIM = 256

    # Fusion parameters
    FUSION_DIM = 256

    # Classifier parameters
    CLASSIFIER_DIMS = [512, 256, 128]
    CLASSIFIER_DROPOUT = 0.4

    # Callbacks
    EARLY_STOPPING_PATIENCE = 15
    REDUCE_LR_PATIENCE = 8

    # Paths
    CHECKPOINT_DIR = "/content/drive/MyDrive/spectrofood_checkpoints"
    RESULTS_DIR = "results"

config = Config()

# Create directories
os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)
os.makedirs(config.RESULTS_DIR, exist_ok=True)



In [6]:
# ============================================================================
# GCS AUTHENTICATION & DATA DOWNLOAD
# ============================================================================
print("\n[1/10] Setting up GCS and downloading data...")

from google.colab import auth
auth.authenticate_user()

storage_client = storage.Client()
bucket = storage_client.bucket(config.GCS_BUCKET)

def download_from_gcs(gcs_path: str, local_path: str) -> None:
    """Download file from GCS to local storage"""
    blob = bucket.blob(gcs_path)
    blob.download_to_filename(local_path)
    size_mb = os.path.getsize(local_path) / (1024**2)
    print(f"  ✓ Downloaded {local_path} ({size_mb:.1f} MB)")

files_to_download = [
    (config.GCS_PREFIX + config.SPATIAL_TRAIN, config.SPATIAL_TRAIN),
    (config.GCS_PREFIX + config.SPATIAL_TEST, config.SPATIAL_TEST),
    (config.GCS_PREFIX + config.SPECTRAL_TRAIN, config.SPECTRAL_TRAIN),
    (config.GCS_PREFIX + config.SPECTRAL_TEST, config.SPECTRAL_TEST),
    (config.GCS_PREFIX + config.LABELS_TRAIN, config.LABELS_TRAIN),
    (config.GCS_PREFIX + config.LABELS_TEST, config.LABELS_TEST),
]

for gcs_path, local_path in files_to_download:
    download_from_gcs(gcs_path, local_path)







[1/10] Setting up GCS and downloading data...
  ✓ Downloaded X_spatial_train.h5 (30328.5 MB)
  ✓ Downloaded X_spatial_test.h5 (7224.2 MB)
  ✓ Downloaded X_spectral_train.npy (250.5 MB)
  ✓ Downloaded X_spectral_test.npy (59.6 MB)
  ✓ Downloaded Y_train.npy (1.8 MB)
  ✓ Downloaded Y_test.npy (0.4 MB)


In [7]:
# ============================================================================
# DATA INSPECTION
# ============================================================================
print("\n[2/10] Inspecting downloaded data...")

X_spectral_train = np.load(config.SPECTRAL_TRAIN)
X_spectral_test = np.load(config.SPECTRAL_TEST)
Y_train = np.load(config.LABELS_TRAIN)
Y_test = np.load(config.LABELS_TEST)

with h5py.File(config.SPATIAL_TRAIN, 'r') as f:
    spatial_train_shape = f['X_spatial'].shape
    print(f"  ✓ Train spatial: {spatial_train_shape}")

with h5py.File(config.SPATIAL_TEST, 'r') as f:
    spatial_test_shape = f['X_spatial'].shape
    print(f"  ✓ Test spatial: {spatial_test_shape}")

print(f"  ✓ Train spectral: {X_spectral_train.shape}")
print(f"  ✓ Test spectral: {X_spectral_test.shape}")
print(f"  ✓ Train labels: {Y_train.shape}")
print(f"  ✓ Test labels: {Y_test.shape}")

assert spatial_train_shape[0] == X_spectral_train.shape[0] == Y_train.shape[0], "Train data misalignment!"
assert spatial_test_shape[0] == X_spectral_test.shape[0] == Y_test.shape[0], "Test data misalignment!"
print(f"  ✓ Data alignment verified")

print(f"\n  Train target (dry matter) distribution:")
print(f"    Mean: {Y_train.mean():.6f}, Std: {Y_train.std():.6f}")
print(f"    Range: [{Y_train.min():.6f}, {Y_train.max():.6f}]")

print(f"\n  Test target (dry matter) distribution:")
print(f"    Mean: {Y_test.mean():.6f}, Std: {Y_test.std():.6f}")
print(f"    Range: [{Y_test.min():.6f}, {Y_test.max():.6f}]")




[2/10] Inspecting downloaded data...
  ✓ Train spatial: (465635, 11, 11, 141)
  ✓ Test spatial: (110844, 11, 11, 141)
  ✓ Train spectral: (465635, 141)
  ✓ Test spectral: (110844, 141)
  ✓ Train labels: (465635,)
  ✓ Test labels: (110844,)
  ✓ Data alignment verified

  Train target (dry matter) distribution:
    Mean: 0.155407, Std: 0.008370
    Range: [0.134966, 0.174347]

  Test target (dry matter) distribution:
    Mean: 0.155359, Std: 0.007759
    Range: [0.140023, 0.170539]


In [8]:
# ============================================================================
# CUSTOM DATA GENERATOR FOR REGRESSION
# ============================================================================
print("\n[3/10] Creating custom data generators...")

class HybridDataGenerator(Sequence):
    """Memory-efficient data generator for hybrid spatial + spectral data (REGRESSION)"""

    def __init__(self,
                 spatial_h5_path: str,
                 spectral_array: np.ndarray,
                 labels: np.ndarray,
                 batch_size: int = 32,
                 shuffle: bool = True,
                 augment: bool = False):
        self.spatial_h5_path = spatial_h5_path
        self.spectral_array = spectral_array
        self.labels = labels
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.augment = augment

        self.spatial_file = h5py.File(spatial_h5_path, 'r')
        self.spatial_dataset = self.spatial_file['X_spatial']

        self.n_samples = len(spectral_array)
        self.indices = np.arange(self.n_samples)

        self._compute_normalization_params()
        self.on_epoch_end()

    def _compute_normalization_params(self):
        """Compute normalization parameters"""
        sample_size = min(1000, self.n_samples)
        sample_indices = np.random.choice(self.n_samples, sample_size, replace=False)
        sample_indices = np.sort(sample_indices)

        spatial_sample = self.spatial_dataset[sample_indices]
        self.spatial_min = spatial_sample.min()
        self.spatial_max = spatial_sample.max()

        self.spectral_min = self.spectral_array.min()
        self.spectral_max = self.spectral_array.max()

        self.label_min = self.labels.min()
        self.label_max = self.labels.max()

    def __len__(self) -> int:
        return int(np.ceil(self.n_samples / self.batch_size))

    def __getitem__(self, index: int) -> Tuple[Dict[str, np.ndarray], np.ndarray]:
        start_idx = index * self.batch_size
        end_idx = min((index + 1) * self.batch_size, self.n_samples)
        batch_indices = self.indices[start_idx:end_idx]

        sorted_positions = np.argsort(batch_indices)
        sorted_indices = batch_indices[sorted_positions]

        spatial_batch = self.spatial_dataset[sorted_indices]
        spatial_batch = spatial_batch[np.argsort(sorted_positions)]

        spectral_batch = self.spectral_array[batch_indices]
        labels_batch = self.labels[batch_indices]

        spatial_batch = (spatial_batch - self.spatial_min) / (self.spatial_max - self.spatial_min + 1e-8)
        spectral_batch = (spectral_batch - self.spectral_min) / (self.spectral_max - self.spectral_min + 1e-8)
        labels_batch = (labels_batch - self.label_min) / (self.label_max - self.label_min + 1e-8)

        if self.augment:
            spatial_batch, spectral_batch = self._augment(spatial_batch, spectral_batch)

        return {
            'spatial_input': spatial_batch.astype(np.float32),
            'spectral_input': spectral_batch.astype(np.float32)
        }, labels_batch.reshape(-1, 1).astype(np.float32)

    def _augment(self, spatial: np.ndarray, spectral: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Data augmentation"""
        for i in range(spatial.shape[0]):
            if np.random.rand() > 0.5:
                spatial[i] = np.flip(spatial[i], axis=0)
            if np.random.rand() > 0.5:
                spatial[i] = np.flip(spatial[i], axis=1)
            if np.random.rand() > 0.75:
                k = np.random.randint(1, 4)
                spatial[i] = np.rot90(spatial[i], k=k, axes=(0, 1))

        noise = np.random.normal(0, 0.01, spectral.shape)
        spectral = spectral + noise
        scale_factor = np.random.uniform(0.95, 1.05, size=(spectral.shape[0], 1))
        spectral = spectral * scale_factor
        spectral = np.clip(spectral, 0, 1)

        return spatial, spectral

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def close(self):
        """Explicitly close HDF5 file"""
        if hasattr(self, 'spatial_file') and self.spatial_file:
            self.spatial_file.close()
            self.spatial_file = None

    def __del__(self):
        self.close()

train_generator = HybridDataGenerator(
    config.SPATIAL_TRAIN,
    X_spectral_train,
    Y_train,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    augment=True
)

val_generator = HybridDataGenerator(
    config.SPATIAL_TEST,
    X_spectral_test,
    Y_test,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    augment=False
)

print(f"  ✓ Train generator: {len(train_generator)} batches")
print(f"  ✓ Val generator: {len(val_generator)} batches")




[3/10] Creating custom data generators...
  ✓ Train generator: 7276 batches
  ✓ Val generator: 1732 batches


In [9]:
# ============================================================================
# MODEL ARCHITECTURE
# ============================================================================
print("\n[4/10] Building hybrid CNN + Transformer model...")

def create_spatial_cnn_branch(input_shape: Tuple[int, int, int]) -> Model:
    inputs = layers.Input(shape=input_shape, name='spatial_input')
    x = inputs

    for i, filters in enumerate(config.CNN_FILTERS):
        x = layers.Conv2D(filters, (3, 3), padding='same', name=f'cnn_conv_{i}')(x)
        x = layers.BatchNormalization(name=f'cnn_bn_{i}')(x)
        x = layers.Activation('relu', name=f'cnn_relu_{i}')(x)
        if i > 0:
            x = layers.Dropout(config.CNN_DROPOUT, name=f'cnn_dropout_{i}')(x)

    x = layers.GlobalAveragePooling2D(name='cnn_gap')(x)
    spatial_features = layers.Dense(config.FUSION_DIM, activation='relu', name='spatial_features')(x)

    return Model(inputs=inputs, outputs=spatial_features, name='spatial_cnn_branch')

def positional_encoding(length: int, depth: int) -> np.ndarray:
    positions = np.arange(length)[:, np.newaxis]
    depths = np.arange(depth)[np.newaxis, :] / depth
    angle_rates = 1 / (10000**depths)
    angle_rads = positions * angle_rates
    pos_encoding = np.concatenate([np.sin(angle_rads), np.cos(angle_rads)], axis=-1)
    return pos_encoding[:, :depth].astype(np.float32)

class TransformerEncoderBlock(layers.Layer):
    def __init__(self, d_model: int, num_heads: int, ff_dim: int, dropout: float = 0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.dropout_rate = dropout

        self.mha = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation='relu'),
            layers.Dense(d_model),
        ])

        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self, inputs, training=False):
        attn_output = self.mha(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.layernorm2(out1 + ffn_output)

        return out2

    def get_config(self):
        config = super().get_config()
        config.update({
            "d_model": self.d_model,
            "num_heads": self.num_heads,
            "ff_dim": self.ff_dim,
            "dropout": self.dropout_rate
        })
        return config

def create_spectral_transformer_branch(input_shape: Tuple[int,]) -> Model:
    inputs = layers.Input(shape=input_shape, name='spectral_input')
    x = layers.Dense(config.TRANSFORMER_DIM, name='spectral_projection')(inputs)
    x = layers.Reshape((1, config.TRANSFORMER_DIM))(x)

    pos_encoding = positional_encoding(1, config.TRANSFORMER_DIM)
    x = layers.Add()([x, pos_encoding])

    for i in range(config.TRANSFORMER_LAYERS):
        x = TransformerEncoderBlock(
            d_model=config.TRANSFORMER_DIM,
            num_heads=config.TRANSFORMER_HEADS,
            ff_dim=config.TRANSFORMER_FF_DIM,
            dropout=0.1,
            name=f'transformer_block_{i}'
        )(x)

    x = layers.GlobalAveragePooling1D(name='transformer_gap')(x)
    spectral_features = layers.Dense(config.FUSION_DIM, activation='relu', name='spectral_features')(x)

    return Model(inputs=inputs, outputs=spectral_features, name='spectral_transformer_branch')

class AttentionFusion(layers.Layer):
    def __init__(self, fusion_dim: int, **kwargs):
        super().__init__(**kwargs)
        self.fusion_dim = fusion_dim

        self.query_dense = layers.Dense(fusion_dim)
        self.key_dense = layers.Dense(fusion_dim)
        self.value_dense = layers.Dense(fusion_dim)
        self.output_dense = layers.Dense(fusion_dim)
        self.layernorm = layers.LayerNormalization(epsilon=1e-6)

        self.reshape_spatial = layers.Reshape((1, fusion_dim))
        self.reshape_spectral = layers.Reshape((1, fusion_dim))

    def call(self, spatial_features, spectral_features):
        spatial_expanded = self.reshape_spatial(spatial_features)
        spectral_expanded = self.reshape_spectral(spectral_features)

        combined = layers.Concatenate(axis=1)([spatial_expanded, spectral_expanded])

        query = self.query_dense(combined)
        key = self.key_dense(combined)
        value = self.value_dense(combined)

        scores = tf.matmul(query, key, transpose_b=True)
        scores = scores / tf.sqrt(tf.cast(self.fusion_dim, tf.float32))
        attention_weights = tf.nn.softmax(scores, axis=-1)

        attended = tf.matmul(attention_weights, value)
        fused = tf.reduce_mean(attended, axis=1)

        output = self.output_dense(fused)
        output = self.layernorm(output)

        return output

    def get_config(self):
        config = super().get_config()
        config.update({"fusion_dim": self.fusion_dim})
        return config

def create_hybrid_model() -> Model:
    """Create hybrid model for REGRESSION"""
    spatial_branch = create_spatial_cnn_branch(config.SPATIAL_SHAPE)
    spectral_branch = create_spectral_transformer_branch(config.SPECTRAL_SHAPE)

    spatial_input = layers.Input(shape=config.SPATIAL_SHAPE, name='spatial_input')
    spectral_input = layers.Input(shape=config.SPECTRAL_SHAPE, name='spectral_input')

    spatial_features = spatial_branch(spatial_input)
    spectral_features = spectral_branch(spectral_input)

    fused_features = AttentionFusion(config.FUSION_DIM, name='attention_fusion')(
        spatial_features, spectral_features
    )

    x = fused_features
    for i, dim in enumerate(config.CLASSIFIER_DIMS):
        x = layers.Dense(dim, activation='relu', name=f'regressor_dense_{i}')(x)
        x = layers.Dropout(config.CLASSIFIER_DROPOUT, name=f'regressor_dropout_{i}')(x)

    outputs = layers.Dense(1, activation='linear', name='output')(x)

    model = Model(
        inputs=[spatial_input, spectral_input],
        outputs=outputs,
        name='hybrid_cnn_transformer_regression'
    )

    return model

# Create model
model = create_hybrid_model()
model.summary()
print(f"  ✓ Model created with {model.count_params():,} parameters")




[4/10] Building hybrid CNN + Transformer model...


Model: "hybrid_cnn_transformer_regression"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ spatial_input       │ (None, 11, 11,    │          0 │ -                 │
│ (InputLayer)        │ 141)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spectral_input      │ (None, 141)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_cnn_branch  │ (None, 256)       │    495,872 │ spatial_input[0]… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spectral_transform… │ (1, 256)          │  1,832,960 │ spectral_input[0… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_fusion    │ (1, 256)          │    263,680 │ spatial_cnn_bran… │
│ (AttentionFusion)   │                   │            │ spectral_transfo… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ regressor_dense_0   │ (1, 512)          │    131,584 │ attention_fusion… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ regressor_dropout_0 │ (1, 512)          │          0 │ regressor_dense_… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ regressor_dense_1   │ (1, 256)          │    131,328 │ regressor_dropou… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ regressor_dropout_1 │ (1, 256)          │          0 │ regressor_dense_… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ regressor_dense_2   │ (1, 128)          │     32,896 │ regressor_dropou… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ regressor_dropout_2 │ (1, 128)          │          0 │ regressor_dense_… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (1, 1)            │        129 │ regressor_dropou… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,888,449 (11.02 MB)

 Trainable params: 2,887,489 (11.01 MB)

 Non-trainable params: 960 (3.75 KB)

  ✓ Model created with 2,888,449 parameters


In [10]:
# ============================================================================
# COMPILE MODEL FOR REGRESSION
# ============================================================================
print("\n[5/10] Compiling model...")

optimizer = keras.optimizers.AdamW(
    learning_rate=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

model.compile(
    optimizer=optimizer,
    loss='mse',
    metrics=[
        'mae',
        keras.metrics.RootMeanSquaredError(name='rmse')
    ]
)

print(f"  ✓ Optimizer: AdamW (LR={config.LEARNING_RATE}, WD={config.WEIGHT_DECAY})")
print(f"  ✓ Loss: Mean Squared Error (MSE)")
print(f"  ✓ Metrics: MAE, RMSE")




[5/10] Compiling model...
  ✓ Optimizer: AdamW (LR=0.0001, WD=1e-05)
  ✓ Loss: Mean Squared Error (MSE)
  ✓ Metrics: MAE, RMSE


In [11]:
# ============================================================================
# CHECKPOINT SYSTEM - PROPER IMPLEMENTATION
# ============================================================================
print("\n[6/10] Setting up checkpoint system...")

# Create checkpoint with all training state
ckpt = tf.train.Checkpoint(
    model=model,
    optimizer=optimizer,
    epoch=tf.Variable(0, dtype=tf.int64),
    best_val_loss=tf.Variable(float('inf'), dtype=tf.float32)
)

# Checkpoint manager
ckpt_manager = tf.train.CheckpointManager(
    ckpt,
    config.CHECKPOINT_DIR,
    max_to_keep=3,
    checkpoint_name='training_ckpt'
)

# Try to restore latest checkpoint
initial_epoch = 0
if ckpt_manager.latest_checkpoint:
    status = ckpt.restore(ckpt_manager.latest_checkpoint)
    status.expect_partial()  # Allow partial restoration if needed
    initial_epoch = int(ckpt.epoch.numpy())
    print(f"✅ Restored checkpoint from epoch {initial_epoch}")
    print(f"   Best validation loss so far: {ckpt.best_val_loss.numpy():.6f}")
    print(f"   Checkpoint: {ckpt_manager.latest_checkpoint}")

    # Verify checkpoint integrity
    print(f"   Verifying restored model...")
    sample_batch = train_generator[0]
    try:
        _ = model(sample_batch[0], training=False)
        print(f"   ✓ Model forward pass successful")
    except Exception as e:
        print(f"   ⚠ Warning: Model verification failed: {e}")
else:
    print("🆕 No checkpoint found - starting from scratch")

print(f"   Training will {'resume from' if initial_epoch > 0 else 'start at'} epoch {initial_epoch}")



[6/10] Setting up checkpoint system...
✅ Restored checkpoint from epoch 14
   Best validation loss so far: 0.071104
   Checkpoint: /content/drive/MyDrive/spectrofood_checkpoints/training_ckpt-14
   Verifying restored model...
   ✓ Model forward pass successful
   Training will resume from epoch 14


In [12]:
# ============================================================================
# PASTE THIS CELL TO REPLACE YOUR EXISTING CALLBACK (Cell 11)
# ============================================================================

print("\n[6.5/10] Setting up callbacks...")

class ImprovedCheckpointCallback(callbacks.Callback):
    """Save checkpoint after each epoch AND save best model separately"""

    def on_epoch_end(self, epoch, logs=None):
        # Update epoch counter
        ckpt.epoch.assign(epoch + 1)

        # Always save checkpoint (training state)
        save_path = ckpt_manager.save()

        # Check if this is the best model
        val_loss = logs.get('val_loss')
        if val_loss is not None and val_loss < ckpt.best_val_loss.numpy():
            # Update best validation loss
            ckpt.best_val_loss.assign(val_loss)

            # Save best model weights - KERAS 3 REQUIRES .weights.h5 !!
            best_model_path = os.path.join(config.CHECKPOINT_DIR, 'best_model.weights.h5')
            self.model.save_weights(best_model_path)

            print(f"\n✅ NEW BEST MODEL - Epoch {epoch + 1} - val_loss: {val_loss:.6f}")
            print(f"   Saved to: {best_model_path}")

# Create callback list
callback_list = [
    ImprovedCheckpointCallback(),

    callbacks.EarlyStopping(
        monitor='val_mae',
        mode='min',
        patience=config.EARLY_STOPPING_PATIENCE,
        restore_best_weights=False,
        verbose=1
    ),

    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        mode='min',
        factor=0.5,
        patience=config.REDUCE_LR_PATIENCE,
        min_lr=1e-7,
        verbose=1
    ),

    callbacks.CSVLogger(
        os.path.join(config.RESULTS_DIR, 'training_log.csv'),
        append=True
    ),

    callbacks.TensorBoard(
        log_dir=os.path.join(config.RESULTS_DIR, 'logs'),
        histogram_freq=1
    )
]

print(f"  ✓ {len(callback_list)} callbacks configured")


[6.5/10] Setting up callbacks...
  ✓ 5 callbacks configured


In [ ]:
# ============================================================================
# TRAINING
# ============================================================================
print("\n[8/10] Starting training...")
print("=" * 80)

try:
    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=config.EPOCHS,
        initial_epoch=initial_epoch,  # THIS IS KEY FOR RESUMING
        callbacks=callback_list,
        verbose=1
    )

    print("=" * 80)
    print("  ✓ Training complete!")

except KeyboardInterrupt:
    print("\n⚠ Training interrupted by user")
    print(f"   Last completed epoch: {int(ckpt.epoch.numpy())}")
    print(f"   Checkpoint saved to: {ckpt_manager.latest_checkpoint}")
    raise
except Exception as e:
    print(f"\n❌ Training failed: {e}")
    raise
finally:
    # Cleanup generators
    print("  Closing data generators...")
    train_generator.close()
    val_generator.close()

# Save final model
final_model_path = os.path.join(config.RESULTS_DIR, 'final_model.weights.h5')
model.save_weights(final_model_path)
print(f"  ✓ Final model saved to: {final_model_path}")

# Save history
with open(os.path.join(config.RESULTS_DIR, 'history.json'), 'w') as f:
    json.dump(history.history, f, indent=2)


[8/10] Starting training...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 8/100
7276/7276 ━━━━━━━━━━━━━━━━━━━━ 2602s 342ms/step - loss: 0.0087 - mae: 0.0639 - rmse: 0.0932 - val_loss: 0.0720 - val_mae: 0.2193 - val_rmse: 0.2683 - learning_rate: 1.0000e-04
Epoch 9/100
7276/7276 ━━━━━━━━━━━━━━━━━━━━ 2546s 350ms/step - loss: 0.0076 - mae: 0.0592 - rmse: 0.0869 - val_loss: 0.0807 - val_mae: 0.2329 - val_rmse: 0.2840 - learning_rate: 1.0000e-04
Epoch 10/100
7276/7276 ━━━━━━━━━━━━━━━━━━━━ 2566s 353ms/step - loss: 0.0067 - mae: 0.0552 - rmse: 0.0819 - val_loss: 0.0752 - val_mae: 0.2193 - val_rmse: 0.2742 - learning_rate: 1.0000e-04
Epoch 11/100
7276/7276 ━━━━━━━━━━━━━━━━━━━━ 2565s 352ms/step - loss: 0.0062 - mae: 0.0525 - rmse: 0.0787 - val_loss: 0.0719 - val_mae: 0.2214 - val_rmse: 0.2681 - learning_rate: 1.0000e-04
Epoch 12/100
7276/7276 ━━━━━━━━━━━━━━━━━━━━ 2582s 355ms/step - loss: 0.0056 - mae: 0.0498 - rmse: 0.0751 - val_loss: 0.0814 - val_mae: 0.2327 - val_rmse: 0.2854 - learning_rate: 1.0000e-04
Epoch 13/100
7276/7276 ━━━━━━━━━━━━━━━━━━━━ 2600s 357ms/s

In [13]:
#============================================================================
# EVALUATION
# ============================================================================
print("\n[9/10] Evaluating on test set...")

best_weights_path = os.path.join(config.CHECKPOINT_DIR, 'best_model.weights.h5')
if os.path.exists(best_weights_path):
    model.load_weights(best_weights_path)
    print(f"  ✓ Loaded best model weights from: {best_weights_path}")

    # Verify weights loaded correctly
    try:
        sample_pred = model.predict(val_generator[0][0], verbose=0)
        print(f"  ✓ Model prediction test passed (shape: {sample_pred.shape})")
    except Exception as e:
        print(f"  ⚠ Warning: Model prediction test failed: {e}")
else:
    print("  ⚠ No best model found, using final weights")

# Evaluate
test_loss, test_mae, test_rmse = model.evaluate(val_generator, verbose=1)

print(f"\n  Test Loss (MSE): {test_loss:.8f}")
print(f"  Test MAE: {test_mae:.8f}")
print(f"  Test RMSE: {test_rmse:.8f}")

# Generate predictions
print("\n  Generating predictions...")
y_pred_norm = model.predict(val_generator, verbose=1)

# Denormalize predictions and targets
y_pred = y_pred_norm.flatten() * (val_generator.label_max - val_generator.label_min) + val_generator.label_min
y_true = Y_test

# Compute metrics on original scale
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print(f"\n  Regression Metrics (Original Scale):")
print(f"    MAE:  {mae:.6f} (dry matter units)")
print(f"    RMSE: {rmse:.6f}")
print(f"    R²:   {r2:.4f}")

# Optional: Convert to 3-class classification for comparison
def classify_prediction(dry_matter):
    """Convert dry matter to 3 classes (for comparison only)"""
    if dry_matter < 0.1516:
        return 0
    elif dry_matter < 0.1594:
        return 1
    else:
        return 2

y_pred_class = np.array([classify_prediction(dm) for dm in y_pred])
y_true_class = np.array([classify_prediction(dm) for dm in y_true])

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
acc = accuracy_score(y_true_class, y_pred_class)

print(f"\n  Classification Accuracy (derived from regression): {acc:.4f}")
print("\n  Classification Report:")
print(classification_report(
    y_true_class,
    y_pred_class,
    target_names=['Unripe', 'Medium', 'Ripe'],
    digits=4
))



[9/10] Evaluating on test set...


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adamw', because it has 2 variables whereas the saved optimizer has 178 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


  ✓ Loaded best model weights from: /content/drive/MyDrive/spectrofood_checkpoints/best_model.weights.h5
  ✓ Model prediction test passed (shape: (64, 1))


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1732/1732 ━━━━━━━━━━━━━━━━━━━━ 36s 19ms/step - loss: 0.0927 - mae: 0.2553 - rmse: 0.3024

  Test Loss (MSE): 0.07110387
  Test MAE: 0.22478843
  Test RMSE: 0.26665309

  Generating predictions...
1732/1732 ━━━━━━━━━━━━━━━━━━━━ 35s 19ms/step

  Regression Metrics (Original Scale):
    MAE:  0.006859 (dry matter units)
    RMSE: 0.008137
    R²:   -0.0997

  Classification Accuracy (derived from regression): 0.3265

  Classification Report:
              precision    recall  f1-score   support

      Unripe     0.3804    0.3157    0.3450     38587
      Medium     0.2474    0.4691    0.3239     33333
        Ripe     0.5362    0.2150    0.3069     38924

    accuracy                         0.3265    110844
   macro avg     0.3880    0.3333    0.3253    110844
weighted avg     0.3951    0.3265    0.3253    110844



In [17]:
# ============================================================================
# INDIVIDUAL VISUALIZATIONS FOR PRESENTATION - GOOGLE DRIVE VERSION
# ============================================================================
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix
from scipy.stats import linregress
from scipy.ndimage import uniform_filter1d
import os

print("\n[10/10] Creating individual visualizations...")

# ============================================================================
# GOOGLE DRIVE CONFIGURATION
# ============================================================================
# Replace 'your_folder_name' with your actual folder name
idc = 'visuals'  # ⚠️ CHANGE THIS to your actual folder name!

# Configure save directory for Google Drive
save_dir = f'/content/drive/MyDrive/{idc}/'

# Create directory if it doesn't exist
try:
    os.makedirs(save_dir, exist_ok=True)
    print(f"✅ Save directory ready: {save_dir}")
except Exception as e:
    print(f"⚠️ Warning: Could not create directory: {e}")
    print(f"   Make sure Google Drive is mounted!")
    print(f"   Run: from google.colab import drive; drive.mount('/content/drive')")
    save_dir = './'  # Fall back to current directory
# ============================================================================

# Set style
plt.style.use('default')
sns.set_palette("husl")

# Calculate metrics once
residuals = y_pred - y_true
errors = np.abs(y_pred - y_true)
within_05 = (errors < 0.005).mean() * 100
within_10 = (errors < 0.010).mean() * 100
cm = confusion_matrix(y_true_class, y_pred_class)
per_class_acc = cm.diagonal() / cm.sum(axis=1)

# ============================================================================
# 1. PREDICTED VS ACTUAL SCATTER PLOT
# ============================================================================
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(y_true, y_pred, alpha=0.4, s=20, c='steelblue', edgecolors='navy', linewidth=0.3)
ax.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()],
        'r--', lw=3, label='Perfect Prediction', zorder=5)

# Add regression line
slope, intercept, r_value, _, _ = linregress(y_true, y_pred)
line_x = np.array([y_true.min(), y_true.max()])
line_y = slope * line_x + intercept
ax.plot(line_x, line_y, 'g-', lw=2.5, alpha=0.7, label=f'Fit: R²={r2:.3f}')

ax.set_xlabel('True Dry Matter (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('Predicted Dry Matter (%)', fontsize=14, fontweight='bold')
ax.set_title(f'Regression Performance\nMAE={mae:.4f}%, R²={r2:.3f}',
             fontsize=16, fontweight='bold', pad=20)
ax.legend(fontsize=12, loc='upper left')
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_axisbelow(True)

# Add text box
textstr = f'Samples: {len(y_true):,}\nMAE: {mae:.4f}%\nRMSE: {rmse:.4f}%\nR²: {r2:.4f}'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=12,
        verticalalignment='top', bbox=props)

plt.tight_layout()
plt.savefig(save_dir + '01_predicted_vs_actual.png', dpi=300, bbox_inches='tight')
plt.savefig(save_dir + '01_predicted_vs_actual.pdf', bbox_inches='tight')
print(f"  ✅ Saved: {save_dir}01_predicted_vs_actual.png")
plt.close()

# ============================================================================
# 2. RESIDUAL PLOT
# ============================================================================
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(y_pred, residuals, alpha=0.4, s=20, c='coral', edgecolors='darkred', linewidth=0.3)
ax.axhline(y=0, color='red', linestyle='--', lw=3, zorder=5, label='Zero Error')
ax.axhline(y=mae, color='orange', linestyle=':', lw=2, alpha=0.7, label=f'+MAE ({mae:.4f}%)')
ax.axhline(y=-mae, color='orange', linestyle=':', lw=2, alpha=0.7, label=f'-MAE ({-mae:.4f}%)')

ax.set_xlabel('Predicted Dry Matter (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('Residuals (%)', fontsize=14, fontweight='bold')
ax.set_title('Residual Plot', fontsize=16, fontweight='bold', pad=20)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(save_dir + '02_residual_plot.png', dpi=300, bbox_inches='tight')
plt.savefig(save_dir + '02_residual_plot.pdf', bbox_inches='tight')
print(f"  ✅ Saved: {save_dir}02_residual_plot.png")
plt.close()

# ============================================================================
# 3. ERROR DISTRIBUTION HISTOGRAM
# ============================================================================
fig, ax = plt.subplots(figsize=(10, 8))
ax.hist(errors, bins=60, color='forestgreen', alpha=0.7, edgecolor='darkgreen', linewidth=1.5)
ax.axvline(mae, color='red', linestyle='--', lw=3, label=f'MAE: {mae:.4f}%')
ax.axvline(np.median(errors), color='blue', linestyle=':', lw=2.5, label=f'Median: {np.median(errors):.4f}%')

ax.set_xlabel('Absolute Error (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=14, fontweight='bold')
ax.set_title('Error Distribution', fontsize=16, fontweight='bold', pad=20)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, axis='y', linestyle='--')
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(save_dir + '03_error_distribution.png', dpi=300, bbox_inches='tight')
plt.savefig(save_dir + '03_error_distribution.pdf', bbox_inches='tight')
print(f"  ✅ Saved: {save_dir}03_error_distribution.png")
plt.close()

# ============================================================================
# 4. CUMULATIVE ERROR DISTRIBUTION
# ============================================================================
fig, ax = plt.subplots(figsize=(10, 8))
sorted_errors = np.sort(errors)
cumulative = np.arange(1, len(sorted_errors) + 1) / len(sorted_errors) * 100

ax.plot(sorted_errors, cumulative, linewidth=3, color='purple')
ax.axvline(0.005, color='green', linestyle='--', lw=2.5, alpha=0.7, label='±0.5% threshold')
ax.axvline(0.010, color='orange', linestyle='--', lw=2.5, alpha=0.7, label='±1.0% threshold')
ax.axhline(50, color='gray', linestyle=':', lw=2, alpha=0.5, label='50th percentile')
ax.axhline(90, color='gray', linestyle=':', lw=2, alpha=0.5, label='90th percentile')

ax.set_xlabel('Absolute Error (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('Cumulative % of Samples', fontsize=14, fontweight='bold')
ax.set_title('Cumulative Error Distribution', fontsize=16, fontweight='bold', pad=20)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_axisbelow(True)

# Add percentiles
textstr = f'Within ±0.5%: {within_05:.1f}%\nWithin ±1.0%: {within_10:.1f}%'
ax.text(0.98, 0.02, textstr, transform=ax.transAxes, fontsize=13,
        verticalalignment='bottom', horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig(save_dir + '04_cumulative_error.png', dpi=300, bbox_inches='tight')
plt.savefig(save_dir + '04_cumulative_error.pdf', bbox_inches='tight')
print(f"  ✅ Saved: {save_dir}04_cumulative_error.png")
plt.close()

# ============================================================================
# 5. CONFUSION MATRIX (3-CLASS)
# ============================================================================
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Unripe', 'Medium', 'Ripe'],
            yticklabels=['Unripe', 'Medium', 'Ripe'],
            cbar_kws={'label': 'Count'},
            annot_kws={'size': 14, 'weight': 'bold'},
            linewidths=2, linecolor='gray')

ax.set_xlabel('Predicted Class', fontsize=14, fontweight='bold')
ax.set_ylabel('True Class', fontsize=14, fontweight='bold')
ax.set_title(f'Confusion Matrix\nOverall Accuracy: {acc*100:.1f}%',
             fontsize=16, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig(save_dir + '05_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.savefig(save_dir + '05_confusion_matrix.pdf', bbox_inches='tight')
print(f"  ✅ Saved: {save_dir}05_confusion_matrix.png")
plt.close()

# ============================================================================
# 6. PER-CLASS ACCURACY BAR CHART
# ============================================================================
fig, ax = plt.subplots(figsize=(10, 8))
class_names = ['Unripe', 'Medium', 'Ripe']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

bars = ax.bar(class_names, per_class_acc * 100, color=colors, alpha=0.8,
              edgecolor='black', linewidth=2)
ax.axhline(y=33.33, color='red', linestyle='--', lw=3, alpha=0.7, label='Random (33.3%)')

# Add value labels on bars
for bar, acc_val in zip(bars, per_class_acc):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 2,
            f'{acc_val*100:.1f}%', ha='center', va='bottom',
            fontsize=14, fontweight='bold')

ax.set_ylabel('Accuracy (%)', fontsize=14, fontweight='bold')
ax.set_title('Per-Class Accuracy', fontsize=16, fontweight='bold', pad=20)
ax.set_ylim(0, 105)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, axis='y', linestyle='--')
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(save_dir + '06_per_class_accuracy.png', dpi=300, bbox_inches='tight')
plt.savefig(save_dir + '06_per_class_accuracy.pdf', bbox_inches='tight')
print(f"  ✅ Saved: {save_dir}06_per_class_accuracy.png")
plt.close()

# ============================================================================
# 7. ERROR VS TRUE VALUE (BIAS CHECK)
# ============================================================================
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(y_true, residuals, alpha=0.3, s=15, c='teal', edgecolors='none')
ax.axhline(y=0, color='red', linestyle='--', lw=3, zorder=5, label='Zero Bias')

# Add moving average trend
sort_idx = np.argsort(y_true)
window = len(y_true) // 20
if window > 1:
    moving_avg = uniform_filter1d(residuals[sort_idx], size=window, mode='nearest')
    ax.plot(y_true[sort_idx], moving_avg, 'orange', lw=4, label='Trend', zorder=4)

ax.set_xlabel('True Dry Matter (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('Residuals (%)', fontsize=14, fontweight='bold')
ax.set_title('Bias Analysis: Error vs True Value', fontsize=16, fontweight='bold', pad=20)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(save_dir + '07_bias_analysis.png', dpi=300, bbox_inches='tight')
plt.savefig(save_dir + '07_bias_analysis.pdf', bbox_inches='tight')
print(f"  ✅ Saved: {save_dir}07_bias_analysis.png")
plt.close()

# ============================================================================
# 8. PREDICTION ACCURACY BY TOLERANCE
# ============================================================================
fig, ax = plt.subplots(figsize=(10, 8))
tolerances = [0.002, 0.005, 0.010, 0.015, 0.020, 0.030]
accuracies = [(errors < tol).mean() * 100 for tol in tolerances]
tolerance_labels = ['±0.2%', '±0.5%', '±1.0%', '±1.5%', '±2.0%', '±3.0%']

bars = ax.barh(tolerance_labels, accuracies, color='skyblue', alpha=0.8,
               edgecolor='navy', linewidth=2)

# Add value labels
for i, (bar, acc_val) in enumerate(zip(bars, accuracies)):
    ax.text(acc_val + 2, i, f'{acc_val:.1f}%',
            va='center', fontsize=13, fontweight='bold')

ax.set_xlabel('% of Samples', fontsize=14, fontweight='bold')
ax.set_title('Prediction Accuracy by Tolerance Level', fontsize=16, fontweight='bold', pad=20)
ax.set_xlim(0, 105)
ax.grid(True, alpha=0.3, axis='x', linestyle='--')
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(save_dir + '08_accuracy_by_tolerance.png', dpi=300, bbox_inches='tight')
plt.savefig(save_dir + '08_accuracy_by_tolerance.pdf', bbox_inches='tight')
print(f"  ✅ Saved: {save_dir}08_accuracy_by_tolerance.png")
plt.close()

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "="*80)
print("📊 ALL VISUALIZATIONS SAVED TO GOOGLE DRIVE")
print("="*80)
print(f"\n📁 Location: {save_dir}")
print("\n✅ Created 8 individual plots:")
print("   1. 01_predicted_vs_actual.png")
print("   2. 02_residual_plot.png")
print("   3. 03_error_distribution.png")
print("   4. 04_cumulative_error.png")
print("   5. 05_confusion_matrix.png")
print("   6. 06_per_class_accuracy.png")
print("   7. 07_bias_analysis.png")
print("   8. 08_accuracy_by_tolerance.png")
print("\n✅ Also saved PDF versions of each")
print("="*80)

# Final summary
print(f"\n📈 Quick Summary:")
print(f"   • MAE: {mae:.4f}% | R²: {r2:.4f}")
print(f"   • Classification Accuracy: {acc*100:.1f}%")
print(f"   • Within ±1%: {within_10:.1f}%")
print("="*80)


[10/10] Creating individual visualizations...
✅ Save directory ready: /content/drive/MyDrive/visuals/
  ✅ Saved: /content/drive/MyDrive/visuals/01_predicted_vs_actual.png
  ✅ Saved: /content/drive/MyDrive/visuals/02_residual_plot.png
  ✅ Saved: /content/drive/MyDrive/visuals/03_error_distribution.png
  ✅ Saved: /content/drive/MyDrive/visuals/04_cumulative_error.png
  ✅ Saved: /content/drive/MyDrive/visuals/05_confusion_matrix.png
  ✅ Saved: /content/drive/MyDrive/visuals/06_per_class_accuracy.png
  ✅ Saved: /content/drive/MyDrive/visuals/07_bias_analysis.png
  ✅ Saved: /content/drive/MyDrive/visuals/08_accuracy_by_tolerance.png

📊 ALL VISUALIZATIONS SAVED TO GOOGLE DRIVE

📁 Location: /content/drive/MyDrive/visuals/

✅ Created 8 individual plots:
   1. 01_predicted_vs_actual.png
   2. 02_residual_plot.png
   3. 03_error_distribution.png
   4. 04_cumulative_error.png
   5. 05_confusion_matrix.png
   6. 06_per_class_accuracy.png
   7. 07_bias_analysis.png
   8. 08_accuracy_by_tolerance.p

In [ ]:
# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("TRAINING COMPLETE - SUMMARY")
print("=" * 80)
print(f"✅ Training epochs completed: {len(history.history['loss'])}")
print(f"✅ Resumed from epoch: {initial_epoch}")
print(f"✅ Best Validation Loss: {ckpt.best_val_loss.numpy():.6f}")
print(f"✅ Final Test MAE: {mae:.6f}")
print(f"✅ Final Test RMSE: {rmse:.6f}")
print(f"✅ R² Score: {r2:.4f}")
print(f"✅ Classification Accuracy (derived): {acc:.4f}")
print(f"✅ Total Parameters: {model.count_params():,}")
print("\n📁 Saved Files:")
print(f"  • Best model: {config.CHECKPOINT_DIR}/best_model.weights.h5")
print(f"  • Final model: {config.RESULTS_DIR}/final_model.weights.h5")
print(f"  • Training log: {config.RESULTS_DIR}/training_log.csv")
print(f"  • History: {config.RESULTS_DIR}/history.json")
print(f"  • Checkpoints: {config.CHECKPOINT_DIR}/")
print("=" * 80)
print("🎉 REGRESSION TRAINING PIPELINE COMPLETE!")
print("=" * 80)